# Check vs postw90 (line)

Compares this repository with postw90 on identical H(R)/A(R) inputs. Run postw90 with `transl_inv_full = true`, `use_ws_distance = true`, `write_aa_r = true`, and `kpath_task = curv` on the generic line `(t, 0.30, 0.15)`, then point `TRIAL03_POSTW90` / `TRIAL04_POSTW90` below at the two output directories. This installs each exported `_r_full.dat` on the consistently pair-mapped Hamiltonian, evaluates the repository curvature at postw90's exact points, and records both the same-model comparison and the trial-set dependence.

Set `RUN_COMPUTATION = False` to skip straight to loading the last saved run from `results/postw90_line/`.

In [ ]:
import sys
from pathlib import Path

import csv
import json
import numpy as np
import matplotlib.pyplot as plt
from pythtb import W90

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from modules import curvature as cv
from wannier.wannier_io import read_wannier_checkpoint
from wannier.position_matrix import (
    apply_orbital_dependent_R_mapping,
    build_position_matrix_from_mmn,
    install_postw90_position_matrix,
)
from validation._paths import RESULTS, STO_TI_Q_2A_RUN
from validation._paths import DATA

PREFIX = "SrTiO3"
N_OCC = 38
COMPONENTS = ((1, 2, "yz"), (2, 0, "zx"), (0, 1, "xy"))
CACHE_NAMES = {
    "trial03": "AA_trial03_48wf_mmn_pair_ws1em05.npz",
    "trial04": "AA_trial04_38wf_mmn_pair_ws1em05.npz",
}

## Config

`TRIAL03_POSTW90` / `TRIAL04_POSTW90` are the postw90 `kpath_task = curv` output directories -- there is no in-repo default, these come from an external postw90 run. `TRIAL03_DIRECTORY` / `TRIAL04_DIRECTORY` override the Wannier source directory (defaults to the Ti_Q_2A trials below); `MMN_CACHE_DIRECTORY`, if set, also compares the repository's MMN-derived A(R) with postw90's.

In [ ]:
RUN_COMPUTATION = False  # True: rebuild both trials against a real postw90 run and recompute. False: load results/postw90_line/.
TRIAL03_POSTW90 = None  # Path to a postw90 kpath_task=curv output dir for trial03; required if RUN_COMPUTATION
TRIAL04_POSTW90 = None  # ... for trial04
TRIAL_DIRECTORIES = {
    "trial03": STO_TI_Q_2A_RUN / "trial_03_Sr_sp_Ti_pd_O_sp/proj_gauge",
    "trial04": DATA / "SrTiO3/Ti_Q_2A/output/188914bc889/trial_04_Sr_sp_Ti_p_O_sp/proj_gauge/188927bc889",
}
MMN_CACHE_DIRECTORY = None  # set to also compare the MMN-derived A(R) with postw90's
OUTPUT_DIR = RESULTS / "postw90_line"

## Helpers

Metric and model-building functions -- pure logic, no side effects.

In [ ]:
def _metrics(reference: np.ndarray, candidate: np.ndarray) -> dict:
    difference = candidate - reference
    result = {
        "all_components_relative_l2": float(
            np.linalg.norm(difference) / np.linalg.norm(reference)
        ),
        "all_components_max_abs_A2": float(np.max(np.abs(difference))),
        "components": {},
    }
    for index, (_, _, name) in enumerate(COMPONENTS):
        reference_norm = np.linalg.norm(reference[:, index])
        result["components"][name] = {
            "relative_l2": float(
                np.linalg.norm(difference[:, index]) / reference_norm
                if reference_norm
                else np.nan
            ),
            "max_abs_A2": float(np.max(np.abs(difference[:, index]))),
            "correlation": float(
                np.corrcoef(reference[:, index], candidate[:, index])[0, 1]
            ),
        }
    return result


def _real_space_metrics(reference: dict, candidate: dict) -> dict:
    """Compare two A(R) dictionaries after both are in the same R convention."""
    reference_R = set(reference)
    candidate_R = set(candidate)
    shared_R = reference_R & candidate_R
    error_squared = np.zeros(3)
    reference_squared = np.zeros(3)
    maximum = np.zeros(3)
    for R in shared_R:
        difference = candidate[R] - reference[R]
        error_squared += np.sum(np.abs(difference) ** 2, axis=(1, 2))
        reference_squared += np.sum(np.abs(reference[R]) ** 2, axis=(1, 2))
        maximum = np.maximum(maximum, np.max(np.abs(difference), axis=(1, 2)))
    return {
        "R_support_exactly_matches": reference_R == candidate_R,
        "R_only_in_candidate": len(candidate_R - reference_R),
        "R_only_in_reference": len(reference_R - candidate_R),
        "relative_frobenius": float(
            np.sqrt(error_squared.sum() / reference_squared.sum())
        ),
        "max_abs_A": float(maximum.max()),
        "components": {
            name: {
                "relative_frobenius": float(
                    np.sqrt(error_squared[index] / reference_squared[index])
                ),
                "max_abs_A": float(maximum[index]),
            }
            for index, name in enumerate(("x", "y", "z"))
        },
    }


def _repository_trace(model, kpoints: np.ndarray) -> np.ndarray:
    """Evaluate the occupied trace with this repository's curvature code."""
    pos = cv.position_terms(model, dtype=np.complex128)
    curvature = cv.omega(cv.connection(cv.fields(model, pos, kpoints), N_OCC))
    reciprocal_inverse = np.linalg.inv(
        np.asarray(model.recip_lat_vecs, dtype=float)
    )
    curvature = np.einsum(
        "ia,jb,ab...->ij...",
        reciprocal_inverse,
        reciprocal_inverse,
        curvature,
    )
    return np.column_stack(
        [
            np.trace(curvature[i, j], axis1=-2, axis2=-1).real
            for i, j, _ in COMPONENTS
        ]
    )


def _mapped_model(directory: Path, checkpoint: dict):
    w90 = W90(str(directory), PREFIX)
    apply_orbital_dependent_R_mapping(
        w90, checkpoint["mp_grid"], verbose=False
    )
    return w90, w90.model(zero_energy=0.0, min_hopping_norm=0.0)


def _evaluate_trial(
    trial: str,
    postw90_directory: Path,
    mmn_cache_directory,
) -> dict:
    directory = TRIAL_DIRECTORIES[trial]
    curvature_path = postw90_directory / f"{PREFIX}-curv.dat"
    position_path = postw90_directory / f"{PREFIX}_r_full.dat"
    raw = np.loadtxt(curvature_path)
    t = raw[:, 0] / raw[-1, 0]
    kpoints = np.column_stack(
        (t, np.full_like(t, 0.30), np.full_like(t, 0.15))
    )
    # postw90 documents the three printed columns as negative Berry curvature.
    postw90_trace = -raw[:, 1:4]

    checkpoint = read_wannier_checkpoint(directory / f"{PREFIX}.chk")
    w90, model = _mapped_model(directory, checkpoint)
    centers_cartesian = (
        np.asarray(w90.lattice.orb_vecs, dtype=float)
        @ np.asarray(w90.lat, dtype=float)
    )
    position_info = install_postw90_position_matrix(
        model,
        position_path,
        centers_cartesian=centers_cartesian,
    )
    repository_trace = _repository_trace(model, kpoints)
    postw90_R = set(model._pos_r)
    result = {
        "t": t,
        "postw90": postw90_trace,
        "repository": repository_trace,
        "same_model_metrics": _metrics(postw90_trace, repository_trace),
        "position_info": {
            "pair_hermiticity_relative": float(
                position_info["pair_hermiticity_relative"]
            ),
            "centres_max_error_A": float(
                position_info["centres_max_error"]
            ),
            "n_R": int(position_info["n_R_pos"]),
        },
    }
    if mmn_cache_directory is not None:
        _, mmn_model = _mapped_model(directory, checkpoint)
        mmn_info = build_position_matrix_from_mmn(
            mmn_model,
            directory,
            PREFIX,
            mmn_directory=directory,
            cache=mmn_cache_directory / CACHE_NAMES[trial],
            translational_invariant_JM=False,
            translational_invariant_MV=False,
            R_distance_tolerance=1e-5,
        )
        mmn_trace = _repository_trace(mmn_model, kpoints)
        mmn_R = set(mmn_model._pos_r)
        result.update(
            {
                "mmn": mmn_trace,
                "construction_metrics": _metrics(repository_trace, mmn_trace),
                "construction_A_R_metrics": _real_space_metrics(
                    model._pos_r, mmn_model._pos_r
                ),
                "mmn_position_info": {
                    "pair_hermiticity_relative": float(
                        mmn_info["mmn_pair_hermiticity_relative"]
                    ),
                    "n_R": int(mmn_info["n_R_pos"]),
                    "zero_H_blocks_added": int(mmn_info["zero_H_blocks_added"]),
                    "weight_outside_ham_R": float(
                        mmn_info["weight_outside_ham_R"]
                    ),
                    "R_support_exactly_matches_postw90": mmn_R == postw90_R,
                    "R_only_in_mmn": len(mmn_R - postw90_R),
                    "R_only_in_postw90": len(postw90_R - mmn_R),
                },
            }
        )
    return result

## Evaluate both trials

`results[trial]["postw90"]` and `results[trial]["repository"]` are the two curvature traces being compared, directly inspectable.

In [ ]:
if RUN_COMPUTATION:
    assert TRIAL03_POSTW90 is not None and TRIAL04_POSTW90 is not None, (
        "set TRIAL03_POSTW90 / TRIAL04_POSTW90 to real postw90 kpath_task=curv output directories"
    )
    results = {
        "trial03": _evaluate_trial("trial03", TRIAL03_POSTW90, MMN_CACHE_DIRECTORY),
        "trial04": _evaluate_trial("trial04", TRIAL04_POSTW90, MMN_CACHE_DIRECTORY),
    }

## Build the summary dict

In [ ]:
if RUN_COMPUTATION:
    summary = {
        "postw90_settings": {
            "transl_inv_full": True,
            "use_ws_distance": True,
            "write_aa_r": True,
            "fermi_energy_eV": 11.0,
            "line_reduced": "(t, 0.30, 0.15), 0 <= t <= 1",
        },
        "trials": {
            key: {
                "same_model_metrics": value["same_model_metrics"],
                "position_info": value["position_info"],
            }
            for key, value in results.items()
        },
        "postw90_trial_comparison": _metrics(
            results["trial03"]["postw90"],
            results["trial04"]["postw90"],
        ),
        "repository_rfull_trial_comparison": _metrics(
            results["trial03"]["repository"],
            results["trial04"]["repository"],
        ),
    }
    if MMN_CACHE_DIRECTORY is not None:
        summary["position_construction_comparison"] = {
            key: {
                "metrics_mmn_relative_to_postw90_rfull": value["construction_metrics"],
                "A_R_metrics_mmn_relative_to_postw90_rfull": value["construction_A_R_metrics"],
                "mmn_position_info": value["mmn_position_info"],
            }
            for key, value in results.items()
        }
        summary["repository_mmn_trial_comparison"] = _metrics(
            results["trial03"]["mmn"],
            results["trial04"]["mmn"],
        )
    print(json.dumps(summary, indent=2))

## Save

In [ ]:
if RUN_COMPUTATION:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_path = OUTPUT_DIR / "postw90_reference_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2) + "\n")
    csv_path = OUTPUT_DIR / "postw90_reference_curves.csv"
    with csv_path.open("w", newline="") as handle:
        writer = csv.writer(handle)
        header = ["t"]
        sources = ["postw90", "repository"]
        if MMN_CACHE_DIRECTORY is not None:
            sources.append("mmn")
        for trial in results:
            for source in sources:
                header.extend(f"{trial}_{source}_{name}_A2" for _, _, name in COMPONENTS)
        writer.writerow(header)
        for index, t in enumerate(results["trial03"]["t"]):
            row = [float(t)]
            for trial in results:
                for source in sources:
                    row.extend(results[trial][source][index].tolist())
            writer.writerow(row)
    print(f"saved {summary_path}")
    print(f"saved {csv_path}")

## Load previously saved results (skip if you just computed above)

Reconstructs `t` and the per-trial `postw90`/`repository` trace arrays straight from the saved CSV, plus `summary` from the JSON.

In [ ]:
if not RUN_COMPUTATION:
    summary = json.loads((OUTPUT_DIR / "postw90_reference_summary.json").read_text())
    with (OUTPUT_DIR / "postw90_reference_curves.csv").open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    t = np.array([float(row["t"]) for row in rows])
    results = {}
    for trial in ("trial03", "trial04"):
        results[trial] = {"t": t}
        for source in ("postw90", "repository"):
            key = f"{trial}_{source}"
            if f"{key}_yz_A2" in rows[0]:
                results[trial][source] = np.array(
                    [[float(row[f"{key}_{name}_A2"]) for _, _, name in COMPONENTS] for row in rows]
                )
    print(json.dumps(summary["trials"], indent=2))

## Plot: postw90 vs repository trace, both trials

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True, dpi=150)
for index, (_, _, name) in enumerate(COMPONENTS):
    ax = axes[index]
    for trial, color in (("trial03", "C0"), ("trial04", "C3")):
        t = results[trial]["t"]
        ax.plot(t, results[trial]["postw90"][:, index], color=color, linewidth=2, label=f"{trial} postw90")
        ax.plot(t, results[trial]["repository"][:, index], color=color, linewidth=1, linestyle="--", label=f"{trial} repository")
    ax.set_ylabel(rf"$\mathrm{{Tr}}\,\Omega_{{{name}}}$ [$\AA^2$]")
    ax.grid(alpha=0.2)
    if index == 0:
        ax.legend(frameon=False, fontsize=7, ncol=2)
axes[-1].set_xlabel(r"$t$ in $\mathbf{k}(t)=(t,0.30,0.15)$")
fig.tight_layout()
plt.show()